# 07 — Deployment, backup & recovery

Operate the system reliably: take consistent backups, verify and restore them, and export metrics. See [docs/deployment.md](../docs/deployment.md) and [docs/backup.md](../docs/backup.md) for the full references.

In [ ]:
import pathlib
import tempfile

from meeting_memory.recovery import BackupManager, export_snapshot, verify_snapshot
from meeting_memory.sdk import MeetingMemoryClient

REPO = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
WORK = pathlib.Path(tempfile.mkdtemp())
DB = WORK / "memory.db"
MeetingMemoryClient.local(DB).import_directory(
    str(REPO / "examples" / "organizations" / "enterprise"), recursive=True
)
print("database ready:", DB)

## Physical backup and restore

`BackupManager` uses SQLite's online backup API for a consistent byte-for-byte copy.

In [ ]:
manager = BackupManager(DB)
manifest = manager.backup(WORK / "backup.db")
print("backed up", manifest.size_bytes, "bytes; checksum", manifest.checksum[:12], "...")

report = manager.restore(WORK / "backup.db")
print("restore ok:", report.ok)

## Logical snapshot

A snapshot is a portable, checksummed JSON export that survives schema upgrades.

In [ ]:
snap_path = WORK / "snapshot.json"
snapshot = export_snapshot(DB, snap_path)
print("wrote", snap_path.name, "-", snapshot.memories, "memories")
print("snapshot verifies:", verify_snapshot(snapshot))

## Export metrics for monitoring

Health metrics can be exported as JSON or Prometheus text. From the CLI:

```bash
meeting-memory metrics --db memory.db --format prometheus
```

In [ ]:
metrics = MeetingMemoryClient.local(DB).metrics()
print("overall health:", round(metrics["overall"], 3))

## Containerized deployment

Build and run the image (see [docs/tutorials/docker-deployment.md](../docs/tutorials/docker-deployment.md)):

```bash
docker build -t meeting-memory:latest .
docker compose up --build
# open http://127.0.0.1:8000/dashboard
```

That completes the notebook tour. See the [tutorials](../docs/tutorials/) for prose walkthroughs of each topic.